# Initial Preprocessing and Data Engineering

In [0]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("SparkSQL_Class")
    .config("spark.sql.pushdown.enabled", "true")
    .getOrCreate()
)

Load churn.csv as a Spark DataFrame

In [0]:
churn_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv("/Workspace/Users/20241747@novaims.unl.pt/big_data_analysis_project")
)

Create a temporary SQL view so both DataFrame operations and SQL queries can be done

In [0]:
churn_df.createOrReplaceTempView("churn_df")

Check DataFrame Schema

In [0]:
churn_df.printSchema()

Change all columns datatypes to the appropriate ones

In [0]:
df_typed = churn_df.selectExpr(
    "try_cast(RowNumber AS INT) AS RowNumber",
    "try_cast(CustomerId AS BIGINT) AS CustomerId",
    "trim(Surname) AS Surname",
    "try_cast(CreditScore AS INT) AS CreditScore",
    "trim(Geography) AS Geography",
    "trim(Gender) AS Gender",
    "try_cast(Age AS INT) AS Age",
    "try_cast(Tenure AS INT) AS Tenure",
    "try_cast(Balance AS DOUBLE) AS Balance",
    "try_cast(NumOfProducts AS INT) AS NumOfProducts",
    "try_cast(HasCrCard AS INT) AS HasCrCard",
    "try_cast(IsActiveMember AS INT) AS IsActiveMember",
    "try_cast(EstimatedSalary AS DOUBLE) AS EstimatedSalary",
    "try_cast(Exited AS INT) AS Exited",
)

df_typed.printSchema()

Check rows and columns count

In [0]:
print(f"Rows: {churn_df.count()}, Columns: {len(churn_df.columns)}")

Use show to preview the first five rows of the DataFrame

In [0]:
churn_df.show(5, truncate=False)

Check summary statistics

In [0]:
churn_df.describe().show(truncate=False)

Check for class imbalance

In [0]:
%sql
SELECT
    Exited,
    COUNT(*) AS customer_count,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct_share
FROM churn_df
GROUP BY Exited
ORDER BY Exited;

There is a clear class imbalance, with class 0 representing 76.57% of the DataFrame and class 1 representing nearly 20%

Drop unnecessary columns:
- RowNumber
- CustomerId
- Surname

In [0]:
cols_to_drop = ["RowNumber", "CustomerId", "Surname"]
churn_df_cleaned = churn_df.drop(*[c for c in cols_to_drop if c in churn_df.columns])

churn_df_cleaned.printSchema()

Check for duplicated rows in the dataset

In [0]:
churn_without_duplicates = churn_df.dropDuplicates()
print(f"Number of duplicates: {churn_df_cleaned.count() - churn_without_duplicates.count()}")

There are 493 rows duplicated in this dataset, these will be dropped

In [0]:
churn_df_cleaned = churn_df_cleaned.dropDuplicates()

Check missing values

In [0]:
from pyspark.sql import functions as F

missing_exprs = [
    F.count(
        F.when(
            F.col(c).isNull()
            | (F.trim(F.col(c).cast("string")) == "")
            | (F.isnan(F.col(c)) if dtype in ("double", "float") else F.lit(False)),
            c,
        )
    ).alias(c)
    for c, dtype in churn_df_cleaned.dtypes
]

churn_df_cleaned.select(missing_exprs).show(vertical=True, truncate=False)

Since the missing values per column represent a negligible percentage of the DataFrame these will dropped

In [0]:
churn_df_cleaned = churn_df_cleaned.dropna(how="any")

Check if all missing values were correctly dropped

In [0]:
from pyspark.sql import functions as F

missing_exprs = [
    F.count(
        F.when(
            F.col(c).isNull()
            | (F.trim(F.col(c).cast("string")) == "")
            | (F.isnan(F.col(c)) if dtype in ("double", "float") else F.lit(False)),
            c,
        )
    ).alias(c)
    for c, dtype in churn_df_cleaned.dtypes
]

churn_df_cleaned.select(missing_exprs).show(vertical=True, truncate=False)

Check for data inconsistencies as: negative balance, age and  number of products, as well as, numbers different from 0 and 1 in columns where these are the only possible values

In [0]:
df_target = churn_df_cleaned  # or df_typed

counts_df = df_target.selectExpr(
    """
    SUM(
        CASE
            WHEN try_cast(Balance AS DOUBLE) < 0
              OR try_cast(Age AS INT) < 0
              OR try_cast(NumOfProducts AS INT) < 0
              OR try_cast(HasCrCard AS INT) NOT IN (0, 1) OR try_cast(HasCrCard AS INT) IS NULL
              OR try_cast(IsActiveMember AS INT) NOT IN (0, 1) OR try_cast(IsActiveMember AS INT) IS NULL
              OR try_cast(Exited AS INT) NOT IN (0, 1) OR try_cast(Exited AS INT) IS NULL
            THEN 1 ELSE 0
        END
    ) AS total_inconsistent_rows
    """,
    "SUM(CASE WHEN try_cast(Balance AS DOUBLE) < 0 THEN 1 ELSE 0 END) AS negative_balance",
    "SUM(CASE WHEN try_cast(Age AS INT) < 0 THEN 1 ELSE 0 END) AS negative_age",
    "SUM(CASE WHEN try_cast(NumOfProducts AS INT) < 0 THEN 1 ELSE 0 END) AS negative_num_of_products",
    "SUM(CASE WHEN try_cast(HasCrCard AS INT) NOT IN (0, 1) OR try_cast(HasCrCard AS INT) IS NULL THEN 1 ELSE 0 END) AS invalid_HasCrCard_not_0_or_1",
    "SUM(CASE WHEN try_cast(IsActiveMember AS INT) NOT IN (0, 1) OR try_cast(IsActiveMember AS INT) IS NULL THEN 1 ELSE 0 END) AS invalid_IsActiveMember_not_0_or_1",
    "SUM(CASE WHEN try_cast(Exited AS INT) NOT IN (0, 1) OR try_cast(Exited AS INT) IS NULL THEN 1 ELSE 0 END) AS invalid_Exited_not_0_or_1",
)

counts_df.show(vertical=True, truncate=False)

No inconsistencies were found